In [1]:
from datasets import load_dataset

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ts_corpus = load_dataset("roneneldan/TinyStories")

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nizwa\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 1350861.12 examples/s]


In [4]:
ts_corpus

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [5]:
ts_corpus["train"][0]["text"]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [3]:
ts_instruct = load_dataset("roneneldan/TinyStoriesInstruct")

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nizwa\.cache\huggingface\hub\datasets--roneneldan--TinyStoriesInstruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 218380/218380 [00:00<00:00, 2005408.11 examples/s]


In [9]:
import re

all_texts = []
for split in ["train", "validation", "test"]:
    for item in ds[split]:
        all_texts.append(item["prompt"])
        all_texts.append(item["utterance"])

# find all patterns like _something_
pattern = re.compile(r"_\w+_")
placeholders = set()

for text in all_texts:
    matches = pattern.findall(text)
    placeholders.update(matches)

print("Found placeholders in dataset:")
print(sorted(placeholders))

Found placeholders in dataset:
['_1_comma_', '_COLON_', '_comma_', '_comma_000_comma_', '_comma_5_comma_', '_comma__comma_', '_comma__comma__comma_', '_comma__comma__comma__comma_', '_comma_cantelope_comma_', '_comma_finally_comma_', '_comma_green_comma_', '_comma_really_comma_']


In [15]:
ds["train"][1]

{'conv_id': 'hit:0_conv:1',
 'utterance_idx': 2,
 'context': 'sentimental',
 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.',
 'speaker_idx': 0,
 'utterance': 'Was this a friend you were in love with_comma_ or just a best friend?',
 'selfeval': '5|5|5_2|2|5',
 'tags': ''}

In [6]:
from easydict import EasyDict

test1 = EasyDict()
test1.name1 = "Test Config 1"

test2 = EasyDict()
test2.name2 = "Test Config 2"

test1.update(test2)

In [7]:
test1

{'name1': 'Test Config 1', 'name2': 'Test Config 2'}

In [21]:
from model import generate_causal_mask
import torch

# 2 batches, 5 tokens each
input_ids = torch.tensor([[1, 0, 0, 2, 3, 4, 5, 0], [6, 7, 8, 9, 10, 0, 0, 0]])

input_ids.shape # (B, L)

torch.Size([2, 8])

In [22]:
print(input_ids)

tensor([[ 1,  0,  0,  2,  3,  4,  5,  0],
        [ 6,  7,  8,  9, 10,  0,  0,  0]])


In [23]:
mask = generate_causal_mask(input_ids.shape[1], input_ids.device)

mask.shape # (1,1,L,L)

torch.Size([1, 1, 8, 8])

In [24]:
num_heads = 4

B, L = input_ids.shape
H = num_heads  # number of attention heads in your MHA

In [25]:
mask = mask.expand(B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [26]:
pad_id = 0

padding_mask = (input_ids != pad_id).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, L)

In [27]:
# True = valid token, False = pad
mask = mask & padding_mask  # (B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [28]:
mask 

tensor([[[[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True,  True,  True, False],
          [ True, False, False,  True,  True,  True,  True, False]],

         [[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True, 

In [ ]:
B, L, V = 1, 4, 5 # batch size, sequence length, vocabulary size

logits = torch.randn(B, L, V)  # (B, L, V)
labels = torch.tensor([[1, 2, 3, 4]])  # (B, L)

In [43]:
shift_logits = logits[:, :-1, :].contiguous()  # (B, L-1, V)
shift_labels = labels[:, 1:].contiguous()      # (B, L-1)

In [45]:
print(logits)

tensor([[[-0.2552,  1.7677, -1.3673, -0.5024, -1.5271],
         [-0.1959,  0.0762, -0.8660, -0.3407,  0.7085],
         [-0.9155,  1.5716, -0.0453,  1.0679, -0.4274],
         [ 1.5407,  1.6160, -1.4435, -0.1791, -0.5408]]])


In [44]:
print(shift_logits)

tensor([[[-0.2552,  1.7677, -1.3673, -0.5024, -1.5271],
         [-0.1959,  0.0762, -0.8660, -0.3407,  0.7085],
         [-0.9155,  1.5716, -0.0453,  1.0679, -0.4274]]])


In [46]:
print(shift_labels)

tensor([[2, 3, 4]])
